# Studio 3.2 — The Diet Problem: Stigler's 1939 Diet

**OPIM 5641 · Business Decision Modeling · Dr. Dave Wanik, UConn**

In 1945, the economist **George Stigler** asked a simple question: what is the cheapest diet that
gives a person every nutrient they need for a year? He had **77 foods** and **9 nutrients**, with 1939
prices.

He couldn't solve it exactly (Simplex hadn't been invented yet), so he used clever trial and error
and got **\$39.93 a year**. A couple of years later, it became one of the first big problems ever
solved with the Simplex method: nine clerks with desk calculators needed about 120 person-days. The
true optimum turned out to be a little lower than Stigler's guess.

Tonight it takes your laptop less than a second.

**How this notebook works.** All of the code is already here. You do not have to write a model
from scratch. Your job is to **run it, read it, and change it**: each experiment has a clearly marked
cell where you plug in a scenario, re-run, and see what happens.

**Before you run anything, guess:** (1) the cheapest cost per year, in 1939 dollars, and (2) how many
of the 77 foods end up in the diet. Write both guesses down.

## Import modules (download Pyomo)

Run this first, every time. It installs Pyomo and the `cbc` solver in Colab.

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES
import pandas as pd

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solver if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

## The model

**A twist in the data.** Each number in Stigler's table is **how much of a nutrient you get for \$1
of that food** (1939 prices). So the decision variable is **how many dollars per day to spend on each
food**, and the cost of the diet is just the total of those dollars.

$$Min\ Z = \sum_{f} x_f$$

subject to, for every nutrient $n$:

$$\sum_{f} (\text{amount of nutrient } n \text{ per dollar of food } f) \cdot x_f \ge \text{daily minimum}_n$$

$$x_f \ge 0$$

That's a **covering** model: minimize cost, cover every requirement (`>=`).

Imagine hard-coding this one: 77 variables, and 9 constraints with 77 terms each. Nobody wants to type
that. This is exactly when you soft-code.

## 1 · The data

The data is the same as the Google OR-Tools Stigler example. We read it from two CSV files.

In [ ]:
# read the data
foods_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/stigler_foods.csv')
req_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/stigler_requirements.csv')

print(len(foods_df), "foods")
foods_df.head()

In [ ]:
# the daily minimums - calories are in THOUSANDS (3 = 3,000 calories)
req_df

Now soft-code it: turn the two tables into dictionaries, the same way we did for Flair's.

In [ ]:
# ---------- 1. THE DATA ----------
# the list of foods and the list of nutrients
foods = list(foods_df['food'])
nutrients = list(req_df['nutrient'])

# daily minimum for each nutrient, plus a readable label
minimum = {}
label = {}
for i in range(len(req_df)):
    nutrient = req_df.loc[i, 'nutrient']
    minimum[nutrient] = float(req_df.loc[i, 'daily_minimum'])
    label[nutrient] = req_df.loc[i, 'label']

# content[food][nutrient] = how much of that nutrient you get for $1 of that food
content = {}
for i in range(len(foods_df)):
    food = foods_df.loc[i, 'food']
    content[food] = {}
    for nutrient in nutrients:
        content[food][nutrient] = float(foods_df.loc[i, nutrient])

print(minimum)
print()
print("Wheat Flour:", content['Wheat Flour (Enriched)'])

# an empty scoreboard - the results cell fills it in
scoreboard = {}

## 2 · The scenario

This is the cell you play with. Leave it alone for the first run.

In [ ]:
# ---------- 2. THE SCENARIO (change things here) ----------
scenario = "1 base"
need = dict(minimum)    # a fresh copy of the daily minimums
banned = []             # foods we refuse to buy
max_share = 1.0         # no single food can be more than this share of the spending (1.0 = no limit)

# Scenario 2: no liver, thank you (remove the # from the next two lines)
# scenario = "2 no liver"
# banned = ['Liver (Beef)']

# Scenario 3: no beans either
# scenario = "3 no liver, no beans"
# banned = ['Liver (Beef)', 'Navy Beans, Dried']

# Scenario 4: more variety - no food can be more than 25% of the spending
# scenario = "4 variety"
# max_share = 0.25

# Scenario 5: an athlete needs 4,000 calories a day
# scenario = "5 athlete"
# need['calories'] = 4.0

# Scenario 6: your own idea goes here


## 3 · The model

Look how short this is. **77 decision variables and 9 nutrient constraints**, and the code is the same
shape as Flair's department loop.

In [ ]:
# ---------- 3. THE MODEL (you never change this cell) ----------
model = ConcreteModel()

# dollars per day to spend on each food (77 of them!)
model.x = Var(foods, domain=NonNegativeReals)

# objective: the total dollars per day
obj_expr = 0
for food in foods:
    obj_expr += model.x[food]

model.cost = Objective(
                      expr = obj_expr,
                      sense = minimize)

# one constraint per nutrient (this loop runs 9 times)
model.constraints = ConstraintList()
for nutrient in nutrients:
    nutrient_expr = 0
    for food in foods:
        nutrient_expr += content[food][nutrient] * model.x[food]
    model.constraints.add(nutrient_expr >= need[nutrient])

# scenario rules: banned foods, and the variety rule
for food in banned:
    model.constraints.add(model.x[food] == 0)
if max_share < 1.0:
    for food in foods:
        model.constraints.add(model.x[food] <= max_share * obj_expr)

# careful - model.pprint() is LONG with 77 foods, so just count instead
print("Number of variables =", len(model.x))
print("Number of constraints =", len(model.constraints))

## 4 · Solve it and read the results

Same printout as the Google OR-Tools example: what to buy for the year, the annual price, and the
nutrients per day against each minimum.

In [ ]:
# ---------- 4. SOLVE AND REPORT (you never change this cell) ----------
results = SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

print("SCENARIO:", scenario)
if results.solver.termination_condition == TerminationCondition.optimal:
    # what to buy for a whole year (only the foods we actually buy)
    print("Annual Foods:")
    for food in foods:
        if model.x[food]() > 0.000001:
            print(food, ": $", round(365 * model.x[food](), 2))

    print()
    print("Optimal annual price: $", round(365 * model.cost(), 4))

    # the first 9 constraints are the nutrients, in the order we added them
    print()
    print("Nutrients per day:")
    k = 1
    for nutrient in nutrients:
        print(label[nutrient], ":", round(model.constraints[k](), 2), "(min", need[nutrient], ")")
        k += 1

    # write it on the scoreboard
    scoreboard[scenario] = round(365 * model.cost(), 2)
else:
    print("INFEASIBLE! No diet can satisfy every rule at the same time. Loosen something.")

**\$39.66 a year** - about 11 cents a day. Stigler's trial-and-error guess was \$39.93, so he was
off by about 27 cents. Not bad for doing it by hand! How close was *your* guess?

The winning diet: **wheat flour, beef liver, cabbage, spinach, and navy beans.** For a year.

Which nutrients are **binding** (exactly at the minimum)? Those are the ones driving the cost. The ones
with room to spare (protein, iron...) come along for free.

### Try it: play with the scenarios

For each scenario: remove the `#` from that scenario's lines in **THE SCENARIO** cell, then run the
scenario cell, the model cell, and the solve cell. Put the `#` back before you try the next one.

| Scenario | What changes | Cost per year | What changed in the diet? |
|---|---|---|---|
| 1 base | nothing | \$39.66 | flour, liver, cabbage, spinach, navy beans |
| 2 | no liver | | |
| 3 | no liver, no navy beans | | |
| 4 | no food more than 25% of the spending | | |
| 5 | 4,000 calories a day | | |
| 6 | your own idea | | |

In [ ]:
# the scoreboard so far
print(scoreboard)

## Talk with your partner

1. Out of 77 foods, the solver picked only 5. Why so few? (Hint: think about corner points. With 9
   constraints, how many variables can be non-zero at a corner?)
2. Would you eat this diet? What is missing from the model?
3. Which scenario cost the most? Why?

## Bottom line

- 77 variables and 9 constraints took the **same code** as 2 variables and 2 constraints. That is
  what soft-coding buys you.
- A scenario is just **more data or one more constraint**. The model cell never changed.
- The cheapest answer is not always an answer you would live with. Adding rules costs money, and the
  model tells you exactly how much.

**Before you move on:** File → Save a copy in GitHub.